# 🚗 EV Driver Behavior Scoring System
## Notebook 4 — Streamlit Dashboard

This notebook writes and launches a **live Streamlit app** inside Google Colab.
The dashboard lets you:
- Input your driving parameters using sliders
- Get your live EV Efficiency Score
- See your driver profile (cluster)
- Get personalized improvement tips based on your weak areas
- View a gauge chart of your score

In [ ]:
# ── Cell 1: Install Dependencies ─────────────────────────────────────────────
!pip install streamlit xgboost shap pyngrok --quiet
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Write the Streamlit App File ─────────────────────────────────────
app_code = '''
import streamlit as st
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import joblib
import os

st.set_page_config(page_title="EV Driver Scorer", page_icon="⚡", layout="wide")

# ── Load models ──────────────────────────────────────────────────────────────
@st.cache_resource
def load_models():
    model  = joblib.load("xgboost_ev_model.pkl")  if os.path.exists("xgboost_ev_model.pkl")  else None
    kmeans = joblib.load("kmeans_driver_clusters.pkl") if os.path.exists("kmeans_driver_clusters.pkl") else None
    scaler = joblib.load("cluster_scaler.pkl")    if os.path.exists("cluster_scaler.pkl")    else None
    return model, kmeans, scaler

model, kmeans, scaler = load_models()

# ── Header ───────────────────────────────────────────────────────────────────
st.title("⚡ EV Driver Efficiency Scorer")
st.markdown("##### Know your driving score and how it affects your EV range")
st.divider()

# ── Sidebar Inputs ───────────────────────────────────────────────────────────
st.sidebar.header("🎮 Your Driving Parameters")
st.sidebar.markdown("Adjust sliders to match your driving style")

avg_speed       = st.sidebar.slider("Avg Speed (km/h)",          10.0, 110.0, 45.0, 1.0)
std_speed       = st.sidebar.slider("Speed Variability (std)",    0.0,  50.0, 15.0, 0.5)
avg_accel       = st.sidebar.slider("Avg Acceleration (m/s²)",   -2.0,   3.0,  0.5, 0.1)
std_accel       = st.sidebar.slider("Acceleration Variability",   0.0,   5.0,  1.0, 0.1)
avg_throttle    = st.sidebar.slider("Avg Throttle Position (%)",  0.0, 100.0, 30.0, 1.0)
avg_brake       = st.sidebar.slider("Avg Brake Pressure",         0.0,  80.0, 15.0, 1.0)
regen_ratio     = st.sidebar.slider("Regen Braking Ratio",        0.1,   0.95, 0.6, 0.01)
aggression      = st.sidebar.slider("Aggression Index",           0.0, 100.0, 35.0, 1.0)
smoothness      = st.sidebar.slider("Speed Smoothness Score",     0.0, 100.0, 70.0, 1.0)
high_speed_r    = st.sidebar.slider("High Speed Ratio (>80 km/h)",0.0,   1.0,  0.2, 0.01)
soc_swing       = st.sidebar.slider("SoC Swing per Trip (%)",     5.0,  60.0, 20.0, 0.5)

# ── Compute Score ─────────────────────────────────────────────────────────────
regen_s  = regen_ratio * 100
aggr_s   = 100 - aggression
speed_s  = (1 - high_speed_r) * 100
ev_score = np.clip(
    regen_s * 0.30 + smoothness * 0.25 + aggr_s * 0.25 + speed_s * 0.20,
    0, 100
)

def get_grade(s):
    if s >= 80: return "A", "Eco Master", "#2196F3"
    elif s >= 65: return "B", "Smooth Commuter", "#4CAF50"
    elif s >= 50: return "C", "Average Driver", "#FF9800"
    elif s >= 35: return "D", "Aggressive Urban", "#FF5722"
    else: return "F", "Energy Waster", "#F44336"

grade, profile, color = get_grade(ev_score)

# Predict kWh/100km
features = np.array([[avg_speed, std_speed, avg_accel, std_accel,
                       avg_throttle, avg_brake, regen_ratio, aggression,
                       smoothness, high_speed_r, soc_swing]])
kwh_pred = float(model.predict(features)[0]) if model else round(
    14 + aggression*0.08 - regen_ratio*3.5 + high_speed_r*4, 2)
est_range = round((40.5 / kwh_pred) * 100, 1)

# Cluster label
cluster_labels = {0: "Eco Master", 1: "Smooth Commuter",
                  2: "Aggressive Urban", 3: "Highway Sprinter"}
if kmeans and scaler:
    cl_feat = scaler.transform([[regen_ratio, aggression, smoothness, high_speed_r, soc_swing]])
    cluster_id = int(kmeans.predict(cl_feat)[0])
    cluster_name = cluster_labels.get(cluster_id, "Unknown")
else:
    cluster_name = profile

# ── Main Dashboard ────────────────────────────────────────────────────────────
col1, col2, col3 = st.columns([1.2, 1, 1])

with col1:
    fig = go.Figure(go.Indicator(
        mode="gauge+number+delta",
        value=round(ev_score, 1),
        delta={"reference": 65, "increasing": {"color": "green"}, "decreasing": {"color": "red"}},
        title={"text": "EV Efficiency Score", "font": {"size": 18}},
        gauge={
            "axis": {"range": [0, 100], "tickwidth": 1},
            "bar": {"color": color},
            "steps": [
                {"range": [0, 35],  "color": "#FFEBEE"},
                {"range": [35, 50], "color": "#FFF3E0"},
                {"range": [50, 65], "color": "#FFF9C4"},
                {"range": [65, 80], "color": "#E8F5E9"},
                {"range": [80, 100],"color": "#E3F2FD"},
            ],
            "threshold": {"line": {"color": "red", "width": 3}, "value": 65}
        }
    ))
    fig.update_layout(height=280, margin=dict(t=40, b=0, l=20, r=20))
    st.plotly_chart(fig, use_container_width=True)

with col2:
    st.markdown(f"### Grade: **{grade}** — {profile}")
    st.markdown(f"**Driver Profile (KMeans):** {cluster_name}")
    st.divider()
    st.metric("Energy Consumption", f"{kwh_pred:.2f} kWh/100km",
              delta=f"{kwh_pred - 14:.2f} vs baseline", delta_color="inverse")
    st.metric("Estimated Range (Nexon EV)", f"{est_range} km")

with col3:
    st.markdown("### Feature Scores")
    metrics = {
        "Regen braking": round(regen_ratio * 100, 1),
        "Smoothness":    round(smoothness, 1),
        "Low aggression":round(aggr_s, 1),
        "Speed control": round(speed_s, 1),
    }
    for label, val in metrics.items():
        st.progress(int(val), text=f"{label}: {val:.0f}/100")

# ── Improvement Tips ─────────────────────────────────────────────────────────
st.divider()
st.markdown("### Personalized Improvement Tips")
tips = []
if regen_ratio < 0.6:
    tips.append("🔋 **Lift off the accelerator earlier** before stops to let regen braking recover energy instead of using friction brakes.")
if aggression > 50:
    tips.append("⚡ **Reduce throttle aggression.** Hard acceleration draws peak current from the battery, degrading it faster and wasting energy.")
if high_speed_r > 0.3:
    tips.append("💨 **Avoid sustained speeds above 80 km/h.** Aerodynamic drag increases with speed squared — your range drops sharply at highway speeds.")
if std_speed > 20:
    tips.append("📈 **Drive at a consistent speed.** Frequent acceleration/deceleration cycles waste energy. Use cruise control where possible.")
if soc_swing > 35:
    tips.append("🔌 **Charge more frequently in smaller amounts.** Keeping SoC between 20-80% reduces battery degradation and improves long-term health.")
if not tips:
    tips.append("✅ Excellent driving! You are maximizing EV range and battery health. Keep it up!")

for tip in tips:
    st.info(tip)

st.divider()
st.caption("Project: EV Driver Behavior Scoring System | Dataset: UCI ECO-driving + Synthetic EV features | Model: XGBoost + KMeans")
'''

with open('ev_dashboard.py', 'w') as f:
    f.write(app_code)

print('✅ ev_dashboard.py written successfully')

In [ ]:
# ── Cell 3: Launch App via ngrok (public URL in Colab) ────────────────────────
# NOTE: Get your free authtoken from https://ngrok.com/
# Replace YOUR_NGROK_TOKEN below with your actual token

NGROK_TOKEN = 'YOUR_NGROK_TOKEN'   # <-- paste your token here

!pip install pyngrok --quiet
from pyngrok import ngrok
import subprocess, time

ngrok.set_auth_token(NGROK_TOKEN)

# Start Streamlit in background
proc = subprocess.Popen(['streamlit', 'run', 'ev_dashboard.py',
                         '--server.port', '8501',
                         '--server.headless', 'true'])
time.sleep(4)

# Create public tunnel
public_url = ngrok.connect(8501)
print('='*50)
print(f'✅ Dashboard is LIVE at: {public_url}')
print('   Open this URL in your browser!')
print('='*50)

## ✅ Notebook 4 Complete!

### How to use the dashboard:
1. Get a free ngrok token at https://ngrok.com (takes 1 min)
2. Paste it in the `NGROK_TOKEN` variable above
3. Run Cell 3 — a public URL will appear
4. Open that URL in your browser
5. Use the sliders to input driving behavior and see the live EV score!

### Project Complete! 🎉
You have built:
- Hybrid real + synthetic EV dataset
- Full EDA with 6 visualization charts
- KMeans driver profiling (4 clusters)
- XGBoost regression with SHAP explainability
- Live Streamlit dashboard with gauge, metrics, and improvement tips